# Experimento 7 — 5 Variáveis, 3 Classificações e Com Balanceamento

A configuração escolhida — **5 variáveis com balanceamento** —  foi selecionada com base na análise comparativa de todos os experimentos anteriores, que avaliaram o número de variáveis (4, 5 e 3) e estratégias de balanceamento.

Dentre todos os cenários testados, o **LGBM com 5 variáveis e balanceamento** se destacou como o mais adequado para o contexto de análise hídrica por reunir as seguintes características:

- **Recall da classe Crítica significativamente superior** em relação aos experimentos sem balanceamento — o que é essencial em um domínio onde deixar passar um caso crítico tem consequências irreversíveis para o meio ambiente e para a gestão dos recursos hídricos;
- **Overfitting consistentemente baixo** (0.010 nos experimentos anteriores com 5 variáveis), indicando boa capacidade de generalização;
- A **redução para 3 classificações** tornou o problema mais tratável e melhorou o desempenho geral do modelo em relação à configuração com 4 rótulos.

Portanto, o objetivo deste experimento é aplicar essa configuração consolidada sobre a **nova amostra rotulada** (`amostra_rotulada_3.parquet`), avaliando se o modelo mantém o bom desempenho observado anteriormente e se a identificação da classe `Crítica` continua melhorada pelo efeito do balanceamento.

A análise será conduzida observando:
- accuracy de treino e de teste;
- diferença entre treino e teste (overfitting);
- precision, recall e F1-score por classe;
- matriz de confusão.


## Preparação do ambiente


In [ ]:
# INSTALAÇÃO DO LIGHTGBM
!pip install lightgbm -q

In [ ]:
# IMPORTAÇÃO DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# CONFIGURAÇÃO DE CAMINHO E CARREGAMENTO
if IN_COLAB:
    print("Ambiente Google Colab detectado.")

    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada_3.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path("../../dataset/amostra_rotulada_3.parquet")

# LEITURA DO DATASET
df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Adequada
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Adequada
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Adequada
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Adequada


In [ ]:
# PREPARANDO O AMBIENTE PARA MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
# DEFINIÇÃO DE X E Y
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "Country",
        "Waterbody Type",
        "Nitrogen (mg/l)"
    ]
]

y = df["conama_status"]

In [ ]:
# TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição das classes no treino:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribuição das classes no teste:")
print(y_test.value_counts(normalize=True).round(4))

Treino: (113119, 5)
Teste: (28280, 5)

Distribuição das classes no treino:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64

Distribuição das classes no teste:
conama_status
Adequada    0.7318
Atenção     0.2585
Crítica     0.0098
Name: proportion, dtype: float64


In [ ]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [ ]:
# CONSTRUÇÃO DO MODELO LGBM COM BALANCEAMENTO
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                class_weight="balanced",
                n_estimators=200,
                learning_rate=0.05,
                num_leaves=31,
                verbose=-1
            )
        )
    ]
)

In [ ]:
# TREINAMENTO
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('classifier',
                 LGBMClassifier(class_weight='balanced', learning_rate=0.05,
                                n_estimators=200, random_state=42,
                                verbose=-1))])

## Avaliação das Métricas de Treino



In [ ]:
y_train_pred = model.predict(X_train)

In [ ]:
train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average='weighted'
)

train_confusion_matrix = confusion_matrix(
    y_train,
    y_train_pred
)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("Train Confusion Matrix:")
print(train_confusion_matrix)

Train Accuracy:
0.754930648255377
Train Precision:
0.817479888302786
Train Recall:
0.754930648255377
Train F1:
0.7798445350264639
Train Confusion Matrix:
[[65923 14043  2809]
 [ 6066 18509  4663]
 [   13   128   965]]


## Avaliação das Métricas de Teste


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# MÉTRICAS DE TESTE — LGBM 5 variáveis com balanceamento
test_accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:")
print(test_accuracy)

print("\nDiferença treino/teste (overfitting):")
print(round(train_accuracy - test_accuracy, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7498939179632249

Diferença treino/teste (overfitting):
0.005

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.91      0.79      0.85     20694
     Atenção       0.56      0.63      0.59      7309
     Crítica       0.09      0.68      0.16       277

    accuracy                           0.75     28280
   macro avg       0.52      0.70      0.53     28280
weighted avg       0.81      0.75      0.78     28280


Confusion Matrix:
[[16427  3567   700]
 [ 1544  4592  1173]
 [    9    80   188]]


## Visualização da Matriz de Confusão


In [ ]:
import plotly.figure_factory as ff

labels = sorted(y_test.unique().tolist())

cm = confusion_matrix(y_test, y_pred, labels=labels)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=[f"Previsto: {l}" for l in labels],
    y=[f"Real: {l}" for l in labels],
    colorscale="Blues",
    showscale=True
)

fig.update_layout(
    title="Matriz de Confusão — LGBM 5 var c/ bal",
    xaxis_title="Classe Prevista",
    yaxis_title="Classe Real",
    width=600,
    height=500
)

fig.show()

## Visualização das Métricas por Classe


In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

classes = [c for c in report.keys() if c not in ["accuracy", "macro avg", "weighted avg"]]

metrics_df = pd.DataFrame(
    {
        "Classe": classes,
        "Precision": [report[c]["precision"] for c in classes],
        "Recall": [report[c]["recall"] for c in classes],
        "F1-score": [report[c]["f1-score"] for c in classes],
    }
)

metrics_melted = metrics_df.melt(
    id_vars="Classe",
    var_name="Métrica",
    value_name="Valor"
)

fig = px.bar(
    metrics_melted,
    x="Classe",
    y="Valor",
    color="Métrica",
    barmode="group",
    title="Precision, Recall e F1-score por Classe — LGBM 5 var c/ bal",
    range_y=[0, 1.05],
    text_auto=".2f"
)

fig.update_layout(width=700, height=450)
fig.show()

## Importância das Variáveis


In [ ]:
lgbm_model = model.named_steps["classifier"]
preprocessor_fitted = model.named_steps["preprocessor"]

ohe_features = preprocessor_fitted.named_transformers_["cat"].get_feature_names_out(categorical_features).tolist()
numeric_features = ["Temperature (cel)", "Orthophosphate (mg/l)", "Nitrogen (mg/l)"]
all_features = ohe_features + numeric_features

importance_df = pd.DataFrame(
    {
        "Variável": all_features,
        "Importância": lgbm_model.feature_importances_
    }
).sort_values("Importância", ascending=True)

fig = px.bar(
    importance_df.tail(20),
    x="Importância",
    y="Variável",
    orientation="h",
    title="Top 20 Variáveis Mais Importantes — LGBM 5 var c/ bal"
)

fig.update_layout(width=700, height=500)
fig.show()

## Resultados Obtidos - Experimento 7

Após a execução do experimento, os principais resultados obtidos com o **LightGBM com 5 variáveis e balanceamento** sobre a nova amostra de 3 rótulos foram consolidados abaixo.

---

### Accuracy e Overfitting

| Métrica | Valor |
|---|---|
| Accuracy treino | 0.754 |
| Accuracy teste | 0.749 |
| Overfitting    | 0.005 |

#### Accuracy

A accuracy de teste indica que o modelo acerta a classificação correta em aproximadamente **75 de cada 100 amostras**. Em um problema com 3 classes desbalanceadas — onde a classe `Adequada` representa a grande maioria dos dados — esse valor precisa ser interpretado com cuidado.

Um modelo "ingênuo" que sempre previsse `Adequada` alcançaria uma accuracy artificialmente alta simplesmente por conta da frequência dessa classe. O fato de o modelo atingir 0.749 enquanto ainda consegue detectar 68% dos casos `Críticos` mostra que ele não está apenas apostando na classe majoritária — ele está genuinamente aprendendo os padrões das três classes.

#### Overfitting baixo

O overfitting mede a diferença entre o que o modelo aprendeu nos dados de treino e o que ele consegue generalizar para dados novos. Uma diferença de **0.005** é praticamente desprezível e indica que:

- O modelo **não memorizou** os dados de treino;
- Os padrões aprendidos são **robustos e transferíveis** para novas amostras;
- O risco de o modelo falhar em produção — quando receber dados que nunca viu — é **muito baixo**.

Dois fatores combinados explicam esse resultado:

O primeiro é o **próprio algoritmo LightGBM**, que utiliza boosting por gradiente com regularização nativa — ao contrário do Random Forest, que cresce árvores independentes e profundas, o LGBM constrói árvores rasas de forma sequencial, corrigindo os erros gradualmente e evitando que o modelo se especialize demais nos dados de treino.

O segundo é o **balanceamento via `class_weight`**, que redistribui o peso das classes durante o aprendizado. Ao forçar o modelo a prestar atenção nas classes minoritárias, o balanceamento impede que ele se otimize exclusivamente para a classe `Adequada` — o que reduziria o overfitting aparente mas tornaria o modelo inútil para os casos mais relevantes.

---

### Métricas por Classe

| Classe | Precision | Recall | F1-score |
|---|---|---|---|
| Adequada | 0.91 | 0.79 | 0.85 |
| Atenção | 0.56 | 0.63 | 0.59 |
| Crítica | 0.09 | 0.68 | 0.16 |


#### Classe `Adequada`

Com precision de **0.91**, recall de **0.79** e F1 de **0.85**, essa é a classe com melhor desempenho, o que era esperado dado o volume maior de amostras.

A precision alta (0.91) significa que quando o modelo classifica um corpo d'água como Adequado, ele está certo em 91% dos casos. O recall um pouco menor (0.79) indica que ele deixa escapar cerca de 21% dos casos realmente adequados, classificando-os como Atenção — um erro tolerável nesse contexto.

---

#### Classe `Atenção`

Com precision de **0.56**, recall de **0.63** e F1 de **0.59**, a classe Atenção apresenta desempenho intermediário.

O recall de 0.63 indica que o modelo identifica a maioria dos casos de atenção real, mas a precision de 0.56 mostra que quase metade das previsões dessa classe são falsas — o modelo tende a classificar como Atenção casos que são Adequados ou Críticos. Isso é típico de classes intermediárias, que compartilham características com as duas extremidades da escala.

---

#### Classe `Crítica`

Essa é a classe mais importante para o domínio hídrico, e o resultado aqui é o **grande destaque do experimento**.

O recall de **0.68** significa que o modelo consegue identificar corretamente 68% dos casos críticos reais. Comparado ao recall de apenas 0.05 registrado nos experimentos sem balanceamento, isso representa uma melhora de mais de 13 vezes na detecção do que realmente importa.

A precision baixa de **0.09** indica que o modelo gera muitos falsos positivos — ou seja, classifica como Crítico casos que não são. No contexto hídrico, isso significa disparar alertas desnecessários em situações normais. Esse é o custo direto do balanceamento, mas é um trade-off justificável: é muito melhor investigar uma situação que não precisava de atenção do que ignorar uma crise ambiental real.

O F1 de **0.16** reflete justamente esse desequilíbrio entre precision e recall, sendo esperado em classes tão minoritárias.


## Análise dos resultados utilizando `class_weight='balanced'`

> *nova nota atualizada

Neste experimento foi utilizado o balanceamento automático disponibilizado pelo próprio LightGBM através do parâmetro `class_weight='balanced'`. O objetivo era reduzir o impacto do forte desbalanceamento existente entre as classes, especialmente devido à baixa quantidade de amostras classificadas como **Crítica**.

Inicialmente, os resultados pareceram promissores, pois o modelo apresentou um aumento significativo do **recall da classe Crítica**, atingindo aproximadamente 68%. Isso significa que o algoritmo passou a identificar uma quantidade muito maior de amostras críticas que anteriormente estavam sendo ignoradas.

Entretanto, ao analisar a **precisão**, observou-se uma queda acentuada, chegando a aproximadamente 9%. Esse comportamento indica que, embora o modelo esteja encontrando mais amostras críticas, ele também está classificando incorretamente muitas amostras de outras classes como sendo críticas.

Em outras palavras, o balanceamento automático acabou atribuindo uma importância excessiva à classe minoritária. Como consequência, o modelo passou a "enxergar" a classe Crítica com muito mais frequência do que deveria, gerando uma grande quantidade de falsos positivos.

A matriz de confusão confirmou esse comportamento. Grande parte das amostras classificadas como Crítica não pertenciam realmente a essa categoria, sendo principalmente amostras originalmente classificadas como Atenção.

Esse resultado sugere que o modelo não está necessariamente aprendendo melhor os padrões da classe Crítica. Na prática, o que está ocorrendo é um deslocamento excessivo da fronteira de decisão em favor dessa classe.

Além disso, a análise mostrou que os principais erros ocorrem entre as classes **Crítica** e **Atenção**, indicando uma possível sobreposição entre essas categorias dentro do conjunto de variáveis disponíveis para treinamento.

Portanto, apesar do aumento expressivo do recall, o balanceamento automático não produziu um equilíbrio adequado entre precisão e recall, tornando necessário o teste de estratégias alternativas de balanceamento.
